In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_=load_dotenv(find_dotenv())
groq_api_key = os.environ["GROQ_API_KEY"]

In [3]:
from langchain_groq import ChatGroq

model = ChatGroq(model="openai/gpt-oss-120b")

### RunnablePassthrough
* It does not do anything to the input data
* Let's see it in a very simple example:a chain with just RunnablePassthrough() will output the original input without any modification.

In [4]:
from langchain_core.runnables import RunnablePassthrough

chain = RunnablePassthrough()

In [5]:
chain.invoke("Abhishekh")

'Abhishekh'

### RunnableLambda
* To use a custom function inside a LCEL chain we need to wrap it up with RunnableLambda
* Let's define a very simple function to create Russian lastnames:

In [6]:
def russian_lastname(name:str) -> str:
    return f"{name}ovich"

In [7]:
from langchain_core.runnables import RunnableLambda

chain = RunnablePassthrough() | RunnableLambda(russian_lastname)

In [8]:
chain.invoke("Bappy")

'Bappyovich'

In [9]:
chain.invoke("Abhi")

'Abhiovich'

### RunnableParallel
* We will use RunnableParallel() for running tasks in parallel.
* This is probably the most important and most useful Runnable in LangChain.
* In the following chain, RunnableParallel is going to run these two tasks in parallel:
    - operation_a will use RunnablePassthrough.
    - operation_b will use RunnableLambda with the russian_lastname function.

In [10]:
from langchain_core.runnables import RunnableParallel

chain = RunnableParallel(
    {
        "operation_a":RunnablePassthrough(),
        "operation_b":RunnableLambda(russian_lastname)
    }
)

In [11]:
chain.invoke("Bappy")

{'operation_a': 'Bappy', 'operation_b': 'Bappyovich'}

In [12]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [13]:
prompt = ChatPromptTemplate.from_template("tell me a curious fact about {soccer_player}")

output_parser = StrOutputParser()

In [ ]:
def russian_lastname_from_dictionary(person):
    return person["name"] + "ovich"

In [15]:
chain = RunnableParallel(
    {
        "operation_a":RunnablePassthrough(),
        "soccer_player":RunnableLambda(russian_lastname_from_dictionary),
        "operation_c":RunnablePassthrough(),
    }
) | prompt | model | output_parser

In [16]:
chain.invoke({
    "name1":"Jordan",
    "name":"Abram"
})

'One especially curious tidbit about Roman\u202fAbramovich is that, long before he became famous worldwide as the owner of Chelsea FC, he was a major player in the post‑Soviet “raider” business—essentially a form of corporate takeover that was almost like a wild‑west version of mergers and acquisitions.\n\nIn the early 1990s, after the Soviet Union collapsed, Russia’s massive state‑owned assets were being privatized. Abramovich, still a relatively unknown 31‑year‑old at the time, teamed up with a group of other savvy (and sometimes notorious) businessmen and used a combination of legal loopholes, aggressive share‑buying tactics, and, according to many insiders, a fair amount of intimidation to wrest control of valuable companies—most notably the oil giant Sibneft.\n\nWhat makes this especially curious is how quickly the move paid off: by the mid‑1990s, Abramovich owned a controlling stake in Sibneft, which later became one of the world’s most profitable oil companies. He eventually sol

Let's see a more advanced use of RunnableParallel

In [19]:
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate 
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

vectorstore = FAISS.from_texts(
    ["dswithbappy focuses on providing content on Data Science, AI, ML, DL, CV, NLP, Python programming, etc. in English."],embedding=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
)
retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:
{context}

Question {question}
"""
prompt = ChatPromptTemplate.from_template(template)

model = ChatGroq(model = "openai/gpt-oss-120b")

retrieval_chain = (
    RunnableParallel({"context":retriever, "question":RunnablePassthrough()})
    | prompt
    | model
    | StrOutputParser()
)

retrieval_chain.invoke("What is dswithbappy?")

'dswithbappy is a source that provides English‑language content covering topics such as Data Science, artificial intelligence, machine learning, deep learning, computer vision, natural language processing, Python programming, and related areas.'

In [22]:
from operator import itemgetter

from langchain_core.prompts import ChatPromptTemplate 
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

vectorstore = FAISS.from_texts(
    ["dswithbappy focuses on providing content on Data Science, AI, ML, DL, CV, NLP, Python programming, etc. in English."],embedding=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
)
retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:
{context}

Question {question}

Answer in the following language:{language}
"""
prompt = ChatPromptTemplate.from_template(template)

model = ChatGroq(model = "openai/gpt-oss-120b")

chain = (
    {
        "context":itemgetter("question") | retriever,
        "question": itemgetter("question"),
        "language":itemgetter("language"),
    }
    | prompt
    | model
    | StrOutputParser()
)

chain.invoke({"question":"What is dswithbappy?","language":"Nepali"})

'dswithbappy एक प्लेटफर्म हो जसले डेटा विज्ञान, कृत्रिम बौद्धिकता (AI), मेसिन लर्निङ (ML), डीप लर्निङ (DL), कम्प्युटर भिजन (CV), नेचुरल ल्याङ्ग्वेज प्रोसेसिङ (NLP), पाइथन प्रोग्रामिङ आदि विषयहरूमा अंग्रेजी भाषामा सामग्री प्रदान गर्नेमा केन्द्रित छ।'